# 📧 Detecting Spam Emails
## Notebook 5: LSTM Neural Network

**KKR Gen AI Innovations** | [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)

---

### 🎯 What You Will Learn
1. How LSTM neural networks work
2. What embeddings are and why they're powerful
3. How to build an LSTM model with Keras
4. How to interpret training curves
5. Final comparison of all three models

---

### 📌 LSTM Intuition

Reading "**not** spam" vs "spam **not** here" — the word ORDER matters!

Regular models (Naive Bayes, SVM) see text as a **bag of words** — word order is lost.
LSTM reads text **sequentially**, like a human reading a sentence, maintaining context.

```
Input: "you have WON a FREE prize"
LSTM:   you → have → WON → a → FREE → prize → [SPAM!]
          ↑     ↑     ↑    ↑     ↑      ↑
          Each word builds on the previous context
```

**This is why LSTM is the most accurate model in this project!**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.utils import download_dataset, compare_models
from src.data_preprocessing import load_dataset, preprocess_dataframe
from src.feature_extraction import split_data, prepare_lstm_sequences
from src.models.lstm_model import (
    build_lstm_model, train_lstm,
    evaluate_lstm, plot_training_history,
    predict_single_email
)

# Check TensorFlow
try:
    import tensorflow as tf
    print(f'✅ TensorFlow {tf.__version__} ready!')
except ImportError:
    print('❌ TensorFlow not installed. Run: pip install tensorflow')

## Step 1: Load and Preprocess Data

In [ ]:
download_dataset('../data')
df = load_dataset('../data/spam.csv')
df = preprocess_dataframe(df)

X = df['cleaned_message']
y = df['label_encoded']
X_train, X_test, y_train, y_test = split_data(X, y)

print('\n✅ Data ready!')

## Step 2: Prepare Sequences for LSTM

LSTM needs text as **integer sequences**:
```
"win free prize" → [142, 87, 203] → [0, 0, ..., 142, 87, 203] (padded to length 100)
```

In [ ]:
MAX_WORDS = 5000   # Vocabulary size
MAX_LEN   = 100    # Sequence length (number of words per message)

X_train_seq, X_test_seq, tokenizer = prepare_lstm_sequences(
    X_train.tolist(), X_test.tolist(),
    max_words=MAX_WORDS,
    max_sequence_length=MAX_LEN
)

# Show a sample
print(f'\nOriginal text:  "{X_train.iloc[0]}"')
print(f'As integers:    {X_train_seq[0][:10]}...')
print(f'Shape:          {X_train_seq[0].shape} (100 integers per message)')

## Step 3: Build the LSTM Model

Let's build our neural network architecture layer by layer.

In [ ]:
model = build_lstm_model(
    vocab_size=MAX_WORDS,
    embedding_dim=64,   # Each word becomes a 64-dimension vector
    max_length=MAX_LEN,
    lstm_units=64,      # 64 LSTM units/neurons
    dropout_rate=0.3    # 30% dropout to prevent overfitting
)

print('\n📊 Model Architecture Summary printed above')
print('   Total parameters = numbers the model will learn during training')

## Step 4: Train the LSTM Model

Training takes longer than Naive Bayes or SVM (a few minutes).
Watch the accuracy improve with each epoch!

In [ ]:
os.makedirs('../results', exist_ok=True)

history = train_lstm(
    model,
    X_train_seq,
    y_train.values,
    epochs=10,
    batch_size=32,
    save_best_to='../results/lstm_best_model.h5'
)

## Step 5: Plot Training History

The training curves show how the model improved over epochs.

In [ ]:
plot_training_history(
    history,
    save_path='../results/lstm_training_history.png'
)

print("""
📖 How to read these curves:
  - Accuracy should INCREASE over epochs
  - Loss should DECREASE over epochs
  - If training >> validation: model is OVERFITTING (memorizing, not learning)
  - If both converge: model is generalizing well ✅
""")

## Step 6: Evaluate the Model

In [ ]:
metrics_lstm = evaluate_lstm(model, X_test_seq, y_test.values)

## Step 7: Predict Custom Emails

In [ ]:
test_emails = [
    "CONGRATULATIONS! You've won a FREE iPhone! CLICK HERE to claim your prize NOW!",
    "Hey, are you free for lunch tomorrow? Let's catch up!",
    "Your account may have been compromised. Verify your password immediately.",
    "Don't forget the team meeting at 3pm. Please review the agenda beforehand.",
    "Earn $5000 per week from home. No skills required. Limited spots available!",
]

print('='*60)
print('📧 LSTM SPAM DETECTION RESULTS')
print('='*60)

for email in test_emails:
    result = predict_single_email(model, tokenizer, email, max_length=MAX_LEN)
    status = '🚨 SPAM' if result['is_spam'] else '✅ HAM '
    print(f'{status} ({result["spam_confidence"]:.1%}) | "{email[:50]}..."')

## Step 8: Final Model Comparison

Let's compare all three models we built in this project!

In [ ]:
# Train Naive Bayes and SVM for comparison
from src.feature_extraction import extract_tfidf_features
from src.models.naive_bayes import train_naive_bayes, evaluate_model as nb_eval
from src.models.svm_model import train_svm, evaluate_svm

X_tr_tfidf, X_te_tfidf, vec = extract_tfidf_features(X_train.tolist(), X_test.tolist())

nb_model = train_naive_bayes(X_tr_tfidf, y_train)
metrics_nb = nb_eval(nb_model, X_te_tfidf, y_test, 'Naive Bayes')

svm_model = train_svm(X_tr_tfidf, y_train)
metrics_svm = evaluate_svm(svm_model, X_te_tfidf, y_test, 'SVM')

In [ ]:
# Final comparison chart
all_metrics = [metrics_nb, metrics_svm, metrics_lstm]
compare_models(all_metrics, save_path='../results/model_comparison.png')

## 🎉 Congratulations! You've Completed the Project!

### What You Built
You built **3 different spam detectors** and compared them:

| Model | Accuracy | Speed | Complexity |
|-------|----------|-------|------------|
| Naive Bayes | ~97% | Very Fast | Low |
| SVM | ~98% | Fast | Medium |
| LSTM | ~99% | Slower | High |

### What You Learned
- ✅ Natural Language Processing (NLP) fundamentals
- ✅ Text preprocessing: cleaning, stopwords, stemming
- ✅ Feature extraction: TF-IDF, word sequences
- ✅ Three ML algorithms: Naive Bayes, SVM, LSTM
- ✅ Model evaluation: accuracy, precision, recall, F1
- ✅ Data visualization: confusion matrices, word clouds

### 🚀 Next Steps
- Try with your own email dataset
- Experiment with different hyperparameters
- Add more features (sender info, subject line)
- Deploy as a web app using Flask or FastAPI

---

### 🏢 About KKR Gen AI Innovations

**KKR Gen AI Innovations** | *Empowering Tomorrow*

- 🌐 Website: [kkrgenaiinnovations.com](https://kkrgenaiinnovations.com/)
- 💬 WhatsApp: [+1 470-861-6312](https://wa.me/14708616312)
- 🐦 Twitter: [@kkr_genai_](https://x.com/kkr_genai_)
- 📘 Facebook: [KKR GenAI Innovations](https://www.facebook.com/kkrgenaiinnovations)
- 📸 Instagram: [@kkrgenaiinnovations](https://www.instagram.com/kkrgenaiinnovations/)
- 💼 LinkedIn: [KKR GenAI Innovations](https://www.linkedin.com/company/kkr-genai-innovations/about/)

---
*Made with ❤️ by KKR Gen AI Innovations*